# Notebook 5 · Adaptive forecasting (skeleton)

> **Status: TO BE COMPLETED.** This notebook is a *framework* only. The data hooks and the model
> interface are sketched out; the bodies marked `TODO` / `NotImplementedError` are left for you to
> fill in.

**Goal — KUN's generalisation across channels.** Notebooks 3-4 fixed the number of channels at
`C = 2` (the insect's `x, y`). The question here: can **one** KUN handle inputs with **different
numbers of channels** (variables) — *automatically aligning channels* — instead of training a
fresh model per dataset?

The plan:

1. build two synthetic datasets with **different** channel counts (e.g. `C = 2` and `C = 3`);
2. define a **channel-adaptive** front-end that maps any `C` into a shared internal width, so the
   same KUN body can consume both;
3. train once, evaluate the model's ability to **transfer / adapt** across channel counts.


In [ ]:
import numpy as np
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.manual_seed(0); np.random.seed(0)
print('Using device:', device)

## 1. Two datasets with different channel counts

A multivariate window has shape `(L, C)`. The whole point is that `C` differs between datasets,
so a fixed `Linear(C, ...)` front-end will not fit both. We make a tiny generator that produces a
trajectory with an arbitrary number of channels.

In [ ]:
def make_multichannel(n=2000, C=2, seed=0):
    """Synthetic (n, C) trajectory: each channel a different mix of sines + noise."""
    rng = np.random.default_rng(seed)
    t = np.arange(n)
    cols = []
    for c in range(C):
        freq = 7 * (c + 1)
        cols.append(np.sin(2*np.pi*t/freq) + 0.1*rng.standard_normal(n))
    return np.stack(cols, axis=1).astype('float32')   # (n, C)

ds_2ch = make_multichannel(C=2, seed=0)
ds_3ch = make_multichannel(C=3, seed=1)
print('dataset A:', ds_2ch.shape, '  dataset B:', ds_3ch.shape)

def make_windows(data, L, H):
    X, Y = [], []
    for i in range(len(data) - L - H + 1):
        X.append(data[i:i+L]); Y.append(data[i+L:i+L+H])
    return np.array(X, 'float32'), np.array(Y, 'float32')

# TODO: build train/test windows + per-channel standardisation for each dataset.

## 2. A channel-adaptive front-end (TO BE COMPLETED)

The idea: project **any** `C` to a shared model width `d_model`, run the **same** KUN body, then
project back to that dataset's `C`. The encoder/decoder body never sees the raw `C`.

Fill in the `TODO`s below.

In [ ]:
class ChannelAdapter(nn.Module):
    """Map (B, L, C) <-> (B, L, d_model) for an arbitrary C.

    TODO: a per-channel-count input projection (C -> d_model) and an
          output projection (d_model -> C). One option: lazily create /
          cache an nn.Linear for each C the model is asked to handle.
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # >>> YOUR CODE HERE <<< : store the projections (e.g. nn.ModuleDict keyed by str(C))
        raise NotImplementedError('TODO: build the channel adapter')

    def encode(self, x):   # (B, L, C) -> (B, L, d_model)
        # >>> YOUR CODE HERE <<<
        raise NotImplementedError

    def decode(self, z, C):  # (B, H, d_model) -> (B, H, C)
        # >>> YOUR CODE HERE <<<
        raise NotImplementedError


class AdaptiveKUN(nn.Module):
    """ChannelAdapter -> shared KUN body -> ChannelAdapter.

    Reuse the KernelUNet from notebook 4 as the body. The body works on a
    fixed d_model, so it is independent of the dataset's channel count.
    """
    def __init__(self, L, H, d_model=32):
        super().__init__()
        # >>> YOUR CODE HERE <<< : self.adapter = ChannelAdapter(d_model)
        #                          self.body    = <KernelUNet operating on d_model channels>
        raise NotImplementedError('TODO: assemble the adaptive KUN')

    def forward(self, x):           # x: (B, L, C) with C possibly varying
        # C = x.size(-1)
        # z = self.adapter.encode(x)        # (B, L, d_model)
        # z = self.body(z)                  # (B, H, d_model)
        # return self.adapter.decode(z, C)  # (B, H, C)
        raise NotImplementedError

## 3. Train & evaluate the adaptation (TO BE COMPLETED)

Train on one channel count, then test whether the same model adapts to another.

Open questions to resolve while implementing:
- Do we **share** the KUN body across channel counts and only swap adapters? (recommended)
- How do we standardise per channel when `C` varies?
- What is the right metric to show "it generalised" — per-channel RMSE on the unseen `C`?

In [ ]:
def train_adaptive(model, loaders, epochs=20, lr=1e-3):
    """TODO: one optimiser; iterate over datasets of different C; Adam + MSE."""
    raise NotImplementedError('TODO: training loop over mixed-channel datasets')


def evaluate_transfer(model, loader):
    """TODO: per-horizon / per-channel RMSE on a dataset with an unseen channel count."""
    raise NotImplementedError('TODO: transfer evaluation')

# TODO: wire datasets A (C=2) and B (C=3) through AdaptiveKUN, train, and report transfer.

## What to finish

- [ ] `ChannelAdapter`: input/output projections that handle an arbitrary `C`.
- [ ] `AdaptiveKUN`: plug in the `KernelUNet` body from notebook 4.
- [ ] Per-channel standardisation that works when `C` differs.
- [ ] Training loop over the mixed-channel datasets.
- [ ] Transfer evaluation: train on `C=2`, adapt to `C=3` (and vice-versa), report per-channel RMSE.

See [Kernel U-Net](https://jiangyou2025.github.io/kun/zh/kernel-u-net/) for the architecture this
builds on.
